# MBA Speedrun: make your study roadmap (runs in Google Colab)

Nothing to install. Click **Runtime → Run all**, or run the cells one by one with the ▶ buttons.

1. **Set up** downloads the project (about 10 seconds).
2. **Your choices**: fill in the form, then run it.
3. **Make the plan** shows the day-by-day plan and lets you download it as a CSV (opens in Excel or Google Sheets).

All hours are **estimates** from the project's topic list and cost parameters, not measured study times.
Source and report: https://github.com/p25prathik-hue/mba-speedrun

In [ ]:
#@title 1. Set up (run once)
import os, subprocess, sys
REPO = "https://github.com/p25prathik-hue/mba-speedrun"
if os.path.exists("../src/speedrun"):          # opened from a downloaded copy of the repo
    os.chdir("..")
elif not os.path.exists("src/speedrun"):
    if not os.path.exists("mba-speedrun"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO + ".git"], check=True)
    os.chdir("mba-speedrun")
sys.path.insert(0, os.path.abspath("src"))
from speedrun import easy
print("Ready. Subject areas in the full IIMA web:")
for k, a in enumerate(easy.areas("iima"), 1):
    print(f"  {k:>2}  {a['area']:<26} {a['topics']:>3} topics  {a['hours']:6.1f} h (estimate)")

In [ ]:
#@title 2. Your choices
#@markdown Type areas as numbers or names from the list above, separated by commas (e.g. `quant, 3`). Leave blank for none.
topic_web = "Full IIMA core (266 topics)"  #@param ["Full IIMA core (266 topics)", "Finance + Accounting pilot (28 topics)"]
areas_i_know_well = ""  #@param {type:"string"}
areas_i_half_know = ""  #@param {type:"string"}
areas_to_skip = ""  #@param {type:"string"}
#@markdown Single topics you already know, by ID (e.g. `QNT01, FIN06`). Run the last cell to see all IDs.
topics_i_know = ""  #@param {type:"string"}
hours_per_day = 6  #@param {type:"number"}
mode = "Full roadmap"  #@param ["Full roadmap", "Crunch: best topics for a fixed number of hours"]
crunch_total_hours = 40  #@param {type:"number"}

web = "iima" if topic_web.startswith("Full") else "pilot"
names = [a["area"] for a in easy.areas(web)]
choices = dict(
    web=web,
    hours_per_day=hours_per_day,
    known_areas=easy.match_areas(areas_i_know_well, names),
    half_known_areas=easy.match_areas(areas_i_half_know, names),
    skip_areas=easy.match_areas(areas_to_skip, names),
    known_topics=easy.match_topics(topics_i_know, web),
)
print("Your choices:", {k: v for k, v in choices.items() if v})

In [ ]:
#@title 3. Make the plan
import time
t0 = time.time()
if mode.startswith("Full"):
    res = easy.make_roadmap(**choices)
else:
    res = easy.make_crunch(crunch_total_hours, **choices)
print(f"Done in {time.time() - t0:.0f} s.\n")
print(easy.summary(res))
print()
print(easy.day_plan_text(res, max_days=10))
path = easy.save_csv(res, f"my_roadmaps/{res['mode']}.csv")
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(res["rows"]))
except ImportError:
    pass
try:
    from google.colab import files
    files.download(path)            # your browser downloads the CSV
except ImportError:
    print("Saved to", os.path.abspath(path))

In [ ]:
#@title (Optional) List every topic ID
cur = None
for t in sorted(easy.topic_list(web), key=lambda t: (t["area"], t["id"])):
    if t["area"] != cur:
        cur = t["area"]; print("\n" + cur)
    print(f"  {t['id']:<6} {t['topic']}")